# UDL Assignment 2 — Part B
## WGAN and WGAN-GP
**Group 56** · **Ankit Kishor Bukshete** · **2025AE05016**  
Add the remaining members of your Assignment 1 group to `config.json` before submission.

**Status:** implementation prepared; full CIFAR-10 training and metrics are not yet run. This notebook and HTML must be executed/exported on the mandatory BITS Prayogshala infrastructure. The current HTML is a source preview, not a completed submission.

## 0. Workflow
Use the same project, configuration, split and hardware as Part A. Finish the full Part A run before generating the combined Part C PDF. Do not change configuration between parts. The two GANs share architecture, initialization and training budgets; their optimizers and Lipschitz constraints follow the respective algorithms.

In [ ]:
from pathlib import Path
import json, sys, platform
assert Path("udl/models.py").is_file(), "Open this notebook from the extracted UDL_Assignment2 project folder."
import torch
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
# If imports fail: run %pip install -r requirements.txt in this kernel and restart.
# Keep the lab's CUDA-compatible torch/torchvision pair when it already works.

In [ ]:
from udl.pipeline import load_config
SMOKE = False  # True only for a small pipeline check; cannot produce final submission metrics.
# First confirm you are in BITS Prayogshala, then set lab_confirmed=true in config.json.
cfg = load_config("config.json", smoke=SMOKE)
print(json.dumps(cfg, indent=2))

## 1. WGAN and WGAN-GP [3 marks]
Both generators map a 128-dimensional normal sample to a 32×32 RGB image with sigmoid output in [0,1]. The critic returns a real score without sigmoid or batch normalization.

WGAN minimizes mean D(fake) − mean D(real) with RMSprop, and clips critic parameters to ±0.01 after every critic step. WGAN-GP uses Adam, without clipping, and adds 10 × mean[(||gradient D(interpolated)||₂−1)²] using per-image interpolations. Both generators minimize −mean D(fake). Five critic steps precede each generator step. Critic updates detach generated images; generator updates freeze critic parameters. The generator stays in training mode during both update types so BatchNorm behavior is consistent. Evaluation mode is used for generated-image evaluation.

A reported GAN epoch contains len(training loader) generator updates and five times as many critic updates. This is more data processing than an autoencoder epoch; final tables report time and cumulative updates explicitly. Compare generated images and FID, not raw loss scales across differently constrained critics.

In [ ]:
from udl.models import Generator, Critic
for model in (Generator(cfg["latent_dim"]),Critic()):
    print(type(model).__name__,"parameters:",sum(p.numel() for p in model.parameters()))
    print(model)
del model

In [ ]:
from udl.pipeline import run_part_b
part_b_results = run_part_b(cfg)

In [ ]:
from udl.reporting import metrics_table, plot_results, review_flags
from IPython.display import display, Image
results = metrics_table(cfg)
columns = ["experiment","fid","training_minutes","sampling_images_per_second","generator_updates","critic_updates"]
display(results[results.experiment.isin(["wgan","wgan_gp"])].reindex(columns=columns).round(4))
for figure in plot_results(cfg):
    if "gan_" in figure.name or "fid_" in figure.name: display(Image(filename=str(figure)))
print("Diagnostics:",review_flags(cfg))

In [ ]:
for name in ("wgan","wgan_gp"):
    print(name,"— exactly 100 unselected unconditional samples")
    display(Image(filename=str(Path(cfg["run_dir"])/name/"samples_100.png")))

## 2. Interpret the results
Replace this paragraph after training with your comparison of stability, FID, diversity, sharpness, artifacts, training time and sampling speed. Explain whether clipping or the gradient penalty appears to affect the outcome. A stable loss alone does not prove realistic samples. Discuss an unexpected ranking honestly and inspect convergence before accepting inadequate training.

Add the GAN visual observations and overall comparison to `observations.json`; these fields are inserted into the Part C PDF. PSNR is N/A for these unconditional GANs because there is no paired reconstruction target.

## 3. Part C — comparative analysis [2 marks]
The report generator requires all nine experiment metrics and inserts four 10×10 grids: VAE beta=1, VQ-VAE K=256 using its PixelCNN prior, WGAN and WGAN-GP. Thus it contains 100 examples of each required model, not 100 examples shared across models. It compares measured reconstruction error, sampling quality and speed. A draft can be created for inspection, but final output requires the visual discussion and real JPG evidence.

In [ ]:
from udl.reporting import build_report, metrics_table
all_results = metrics_table(cfg, require_all=True)
display(all_results.reindex(columns=["experiment","fid","psnr","mse","training_minutes","sampling_images_per_second"]).round(4))
draft_pdf = build_report(cfg, final=False)
print("Inspect the draft, edit observations.json, add the lab JPG, then finalize.")

## 4. Required actions before finalization
1. Capture your actual BITS Prayogshala page showing the running assignment and allocated GPU. Save a genuine JPEG as `BITS_lab_usage.jpg`.
2. Fill `observations.json` with observations based on the actual four grids and metrics. Include all group members in `config.json`.
3. Replace the interpretation paragraphs in both notebooks, save both with outputs, and export them.
4. From the project terminal, run `python finalize_submission.py`. It checks saved output cells, final metrics, the screenshot and discussion, exports both HTML files and creates the final PDF and submission ZIP.
5. Review the ZIP and submit once for Group 56. The screenshot must show real lab usage; the environment log alone does not satisfy the requirement.

In [ ]:
import subprocess
if Path("UDL_Group56_PartB.ipynb").is_file():
    subprocess.run([sys.executable,"-m","nbconvert","--to","html","UDL_Group56_PartB.ipynb"],check=True)
else:
    print("Save/download the notebook into the project folder before exporting.")